# Demo 2: Temporal Forensics — VLM-Powered Video Audit

**Hook:** Your world model generated 1,000 videos. Can a VLM audit them automatically?

**What this demo shows:**
- Derive temporal ground truth from existing frame-level annotations
- Run a real VLM (Qwen3-VL-2B) to independently localize events in each video
- Evaluate VLM predictions against ground truth using temporal IoU
- Find disagreements and build a clip-level review queue

**Duration:** ~5 minutes | **Dataset:** `quickstart-video` (10 dashcam clips with bounding box annotations)

## 1. Load dataset & derive ground truth

The videos have per-frame bounding box labels (`person`, `vehicle`, `road_sign`).
We convert these to temporal events: if `person` appears in frames 30-90, that
becomes a `person_visible` event spanning those frames. This gives us real
ground truth derived from real annotations — not fabricated data.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F
import numpy as np
import json

dataset = foz.load_zoo_dataset("quickstart-video")
dataset.compute_metadata()
print(f"Loaded {len(dataset)} video samples")

# Derive temporal ground truth from frame-level bounding box annotations
for sample in dataset:
    fps = sample.metadata.frame_rate
    fc = sample.metadata.total_frame_count

    # Collect which frames each label appears in
    label_frames = {}
    for fn, frame in sample.frames.items():
        if frame.detections:
            for det in frame.detections.detections:
                # Normalize: "road sign" -> "road_sign" for consistent matching
                label = det.label.replace(" ", "_") + "_visible"
                label_frames.setdefault(label, []).append(int(fn))

    # Convert to temporal events (merge frames within 3-frame gaps)
    gt_events = []
    for label, flist in label_frames.items():
        flist.sort()
        start = flist[0]
        prev = flist[0]
        for f in flist[1:]:
            if f - prev > 3:  # gap > 3 frames = new event
                if prev - start >= 5:
                    gt_events.append(fo.TemporalDetection(
                        label=label, support=[start, prev]))
                start = f
            prev = f
        if prev - start >= 5:
            gt_events.append(fo.TemporalDetection(
                label=label, support=[start, prev]))

    sample["ground_truth_events"] = fo.TemporalDetections(detections=gt_events)
    sample.save()

    labels = [f"{e.label}[{e.support[0]}-{e.support[1]}]" for e in gt_events]
    print(f"  {sample.filepath.split('/')[-1]}: {labels}")

print(f"\nTotal GT events: {dataset.count('ground_truth_events.detections')}")

## 2. Run VLM temporal localization

We run Qwen3-VL-2B on each video — 8 sampled frames per clip — and ask it to
identify visible objects with start/end timestamps. The VLM has never seen the
ground truth annotations. This is a genuine, independent audit.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info
from decord import VideoReader, cpu as decord_cpu
from PIL import Image

print("Loading Qwen3-VL-2B-Instruct...")
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen3-VL-2B-Instruct", trust_remote_code=True)
vlm = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3-VL-2B-Instruct",
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
print(f"Model loaded — {torch.cuda.memory_allocated() / 1e9:.1f}GB GPU memory")

In [ ]:
def run_vlm_on_video(sample, processor, vlm):
    """Run VLM temporal event localization on a single video."""
    fc = sample.metadata.total_frame_count
    fps = sample.metadata.frame_rate
    duration = fc / fps

    # Extract 8 evenly-spaced frames
    vr = VideoReader(sample.filepath, ctx=decord_cpu(0))
    indices = np.linspace(0, len(vr) - 1, 8, dtype=int)
    frames = [Image.fromarray(vr[i].asnumpy()) for i in indices]

    # Prompt the VLM with the same label vocabulary as ground truth
    messages = [{
        "role": "user",
        "content": [
            *[{"type": "image", "image": frame} for frame in frames],
            {"type": "text", "text": (
                f"These are 8 evenly-spaced frames from a {duration:.1f}-second video. "
                f"List each distinct visible object or activity. For each, estimate "
                f"when it is first visible (start) and last visible (end) in seconds. "
                f"Use labels like: person_visible, vehicle_visible, road_sign_visible. "
                f'Format as JSON: [{{"label": "...", "start": N.N, "end": N.N}}]'
            )},
        ],
    }]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(vlm.device)

    with torch.no_grad():
        output_ids = vlm.generate(**inputs, max_new_tokens=512, do_sample=False)

    generated = output_ids[0][inputs.input_ids.shape[1]:]
    response = processor.decode(generated, skip_special_tokens=True)

    # Parse JSON response into TemporalDetections
    vlm_events = []
    try:
        json_start = response.find("[")
        json_end = response.rfind("]") + 1
        if json_start >= 0 and json_end > json_start:
            events = json.loads(response[json_start:json_end])
            for evt in events:
                start_frame = max(1, int(float(evt["start"]) * fps) + 1)
                end_frame = min(fc, int(float(evt["end"]) * fps) + 1)
                label = str(evt["label"]).lower().strip()
                if end_frame > start_frame:
                    vlm_events.append(fo.TemporalDetection(
                        label=label, support=[start_frame, end_frame]))
    except (json.JSONDecodeError, ValueError, KeyError):
        pass

    return vlm_events, response

print("VLM inference function ready.")

In [ ]:
# Run VLM on every video
import time

print("Running VLM on each video...")
t0 = time.time()
for i, sample in enumerate(dataset):
    fname = sample.filepath.split("/")[-1]
    vlm_events, response = run_vlm_on_video(sample, processor, vlm)

    sample["vlm_events"] = fo.TemporalDetections(detections=vlm_events)
    sample["vlm_raw_response"] = response
    sample.save()

    vlm_labels = [e.label for e in vlm_events]
    print(f"  [{i+1}/{len(dataset)}] {fname}: {vlm_labels}")

elapsed = time.time() - t0
print(f"\nProcessed {len(dataset)} videos in {elapsed:.1f}s ({elapsed/len(dataset):.1f}s/video)")
print(f"Total VLM events: {dataset.count('vlm_events.detections')}")

## 3. Evaluate: temporal IoU

Now we compare the VLM's predictions against the derived ground truth using
**temporal IoU** — the same metric used in ActivityNet benchmarks. This tells us:
- **True positives:** events the VLM correctly found (matching label + sufficient temporal overlap)
- **False negatives:** events the VLM missed entirely
- **False positives:** events the VLM hallucinated or mislocalized

In [ ]:
# Ground truth and prediction field names
gt_field = "ground_truth_events"
pred_field = "vlm_events"
print(f"Ground truth field: {gt_field}")
print(f"Prediction field:   {pred_field}")

In [ ]:
# Evaluate VLM predictions against ground truth using temporal IoU (idempotent)
eval_key = "temporal_eval"
if eval_key in dataset.list_evaluations():
    dataset.delete_evaluation(eval_key)

import warnings
warnings.filterwarnings("ignore", message=".*unique classes.*")

results = dataset.evaluate_detections(
    pred_field,
    gt_field=gt_field,
    eval_key=eval_key,
)

results.print_report()

# Summary counts
tp = dataset.sum(f"{eval_key}_tp")
fp = dataset.sum(f"{eval_key}_fp")
fn = dataset.sum(f"{eval_key}_fn")
print(f"\nTotal: {tp:.0f} TP, {fp:.0f} FP, {fn:.0f} FN")
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
print(f"Precision: {precision:.2f}, Recall: {recall:.2f}")

## 4. Find disagreements

The most valuable output isn't the mAP number — it's *where* the VLM disagrees
with ground truth. These disagreement regions are exactly what a human reviewer
needs to look at.

In [ ]:
# Find disagreement samples using evaluation results
# After evaluate_detections, each detection gets a "{eval_key}" field: "tp", "fp", or "fn"
missed_ids = []
hallucinated_ids = []

for sample in dataset:
    # Check GT for false negatives (events the VLM missed)
    gt = sample[gt_field]
    if gt is not None:
        for det in gt.detections:
            if det[eval_key] == "fn":
                missed_ids.append(sample.id)
                break

    # Check predictions for false positives (events the VLM hallucinated)
    preds = sample[pred_field]
    if preds is not None:
        for det in preds.detections:
            if det[eval_key] == "fp":
                hallucinated_ids.append(sample.id)
                break

missed = dataset.select(missed_ids) if missed_ids else dataset.limit(0)
hallucinated = dataset.select(hallucinated_ids) if hallucinated_ids else dataset.limit(0)

print(f"Samples with missed events (FN):       {len(missed)}")
print(f"Samples with hallucinated events (FP):  {len(hallucinated)}")

all_disagree_ids = list(set(missed_ids + hallucinated_ids))
disagreements = dataset.select(all_disagree_ids) if all_disagree_ids else dataset
print(f"\nTotal disagreement samples: {len(disagreements)}")
print("These are your priority review queue.")

## 5. Launch the App — timeline visualization

**What to do in the App:**
1. Look at the video player — you'll see **dual timeline bars**:
   - Green: ground truth events
   - Blue: VLM predictions
2. Spot disagreements visually — where bars don't overlap
3. Click on a disagreement region to jump to that timestamp

In [ ]:
# Launch App showing disagreement samples (bound to 0.0.0.0 for remote access)
session = fo.launch_app(disagreements, port=5151, address="0.0.0.0")
print("App launched with disagreement samples")
print(f"\n>> {len(disagreements)} videos with VLM errors — look for misaligned timeline bars")
print(">> Green bars = ground truth, Blue bars = VLM predictions")

## 6. Create reviewable clips from disagreement regions

`to_clips()` is the power move: it converts the full video dataset into a
clip-level dataset where each clip corresponds to one VLM-predicted event.
Now reviewers see *just* the relevant segment, not the whole video.

In [ ]:
# Create clip-level view from VLM events — each clip = one predicted event
clips = dataset.to_clips("vlm_events")

print(f"Created {len(clips)} reviewable clips from VLM predictions")
for clip in clips:
    label = clip.vlm_events.label
    support = clip.support
    eval_result = clip.vlm_events[eval_key]
    marker = "FP!" if eval_result == "fp" else "ok"
    print(f"  [{support[0]:>5d} - {support[1]:>5d}] {label:<25s} {marker}")

# Show clips in the App
session.view = clips
print("\n>> App now shows individual clips — FP clips are your hallucinations")

## Takeaway

**A validation pipeline you can build in an afternoon, not a quarter.**

What we just built — with real data, no fabrication:
1. Derived temporal ground truth from existing frame-level annotations
2. Ran a real VLM (Qwen3-VL-2B) as an independent auditor
3. Quantitative evaluation with temporal IoU — real TP/FP/FN counts
4. Automatic disagreement detection → prioritized review queue
5. Clip-level review so reviewers see only the moments that matter

This same pipeline works for auditing world model outputs:
- Ground truth = what *should* happen given the scenario specification
- VLM = automated auditor checking what *actually* happens in generated video
- Disagreements = where your world model is wrong, ranked for human review